<a href="https://colab.research.google.com/github/genairsh/GenAI_Level3_Assignment/blob/main/silver/Recipe_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import warnings
# Suppress FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [21]:
# Install dependencies
!pip -q install pandas numpy scikit-learn matplotlib seaborn sentence-transformers faiss-cpu openai

In [25]:
# Install the new Google GenAI SDK
!pip install -q google-genai

In [ ]:
# -------------------------------
# 1. Imports
# -------------------------------
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import re
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss



In [ ]:
#Load recipes
df = pd.read_csv("https://raw.githubusercontent.com/genairsh/GenAI_Level3_Assignment/main/datasets/recipes.csv")
print("Dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())

Dataset shape: (50, 13)


,recipe_id,recipe_name,ingredients,dietary_tags,difficulty,cooking_time_minutes,cuisine,taste,calories,protein_g,carbs_g,fat_g,instructions
0,1,Vegetable Fried Rice,"rice,onion,carrot,peas,garlic,soy sauce","vegan,vegetarian",easy,25,Asian,"savory,spicy",420,9,68,12,"Cook rice. Stir-fry garlic and vegetables, add..."
1,2,Chickpea Curry,"chickpeas,onion,tomato,garlic,ginger,curry powder","vegan,vegetarian,gluten-free",easy,30,Indian,"spicy,savory",380,15,52,11,"Saute onion, garlic and ginger. Add spices and..."
2,3,Paneer Butter Masala,"paneer,tomato,onion,butter,cream,garam masala","vegetarian,gluten-free",medium,40,Indian,"rich,creamy,savory",520,21,24,36,"Prepare a tomato-onion gravy, add spices and p..."
3,4,Lemon Garlic Pasta,"pasta,garlic,lemon,olive oil,parsley","vegan,vegetarian",easy,20,Italian,"fresh,tangy,savory",410,11,64,13,"Cook pasta. Saute garlic in olive oil, add lem..."
4,5,Vegetable Quinoa Bowl,"quinoa,cucumber,tomato,carrot,chickpeas,lemon","vegan,vegetarian,gluten-free",easy,25,Mediterranean,"fresh,tangy",390,14,58,11,Cook quinoa and combine with chopped vegetable...


['recipe_id', 'recipe_name', 'ingredients', 'dietary_tags', 'difficulty', 'cooking_time_minutes', 'cuisine', 'taste', 'calories', 'protein_g', 'carbs_g', 'fat_g', 'instructions']


In [ ]:
# ----------------------------------------------
# 3. Data quality checks
# ----------------------------------------------
required_columns = [
    "recipe_id", "recipe_name", "ingredients", "dietary_tags",
    "difficulty", "cooking_time_minutes", "cuisine", "taste",
    "calories", "protein_g", "carbs_g", "fat_g", "instructions"
]

missing_columns = [c for c in required_columns if c not in df.columns]
assert not missing_columns, f"Missing columns: {missing_columns}"

print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

text_columns = [
    "recipe_id", "recipe_name", "ingredients", "dietary_tags",
    "difficulty", "cuisine", "taste", "instructions"
]
for c in text_columns:
    df[c] = df[c].fillna("").astype(str)

numeric_columns = [
    "cooking_time_minutes", "calories", "protein_g", "carbs_g", "fat_g"
]
for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Safety check: some CSV files may not have avg_rating
if 'avg_rating' not in df.columns:
    df['avg_rating'] = 4.0
else:
    df['avg_rating'] = df['avg_rating'].fillna(4.0)

print('Cleaned dataset preview:')
display(df[['recipe_name', 'ingredients', 'dietary_tags', 'difficulty']].head())

recipe_id               0
recipe_name             0
ingredients             0
dietary_tags            0
difficulty              0
cooking_time_minutes    0
cuisine                 0
taste                   0
calories                0
protein_g               0
carbs_g                 0
fat_g                   0
instructions            0
dtype: int64
Duplicate rows: 0
Cleaned dataset preview:


,recipe_name,ingredients,dietary_tags,difficulty
0,Vegetable Fried Rice,"rice,onion,carrot,peas,garlic,soy sauce","vegan,vegetarian",easy
1,Chickpea Curry,"chickpeas,onion,tomato,garlic,ginger,curry powder","vegan,vegetarian,gluten-free",easy
2,Paneer Butter Masala,"paneer,tomato,onion,butter,cream,garam masala","vegetarian,gluten-free",medium
3,Lemon Garlic Pasta,"pasta,garlic,lemon,olive oil,parsley","vegan,vegetarian",easy
4,Vegetable Quinoa Bowl,"quinoa,cucumber,tomato,carrot,chickpeas,lemon","vegan,vegetarian,gluten-free",easy


In [ ]:
# ----------------------------------------------
# 4. Normalize ingredients
# ----------------------------------------------
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9,\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
SYNONYMS = {
    "capsicum": "bell pepper",
    "bell peppers": "bell pepper",
    "tomatoes": "tomato",
    "onions": "onion",
    "potatoes": "potato"
}

def normalize_ingredient(item):
    item = normalize_text(item).strip()
    return SYNONYMS.get(item, item)

def ingredient_set(text):
    return {
        item.strip()
        for item in text.split(",")
        if item.strip()
    }

In [ ]:
df["ingredients_clean"] = df["ingredients"].apply(normalize_ingredient)
df["ingredient_set"] = df["ingredients_clean"].apply(ingredient_set)

print("\nNormalized sample:\n")
display(df[["recipe_name", "ingredients_clean", "ingredient_set"]].head())




Normalized sample:



,recipe_name,ingredients_clean,ingredient_set
0,Vegetable Fried Rice,"rice,onion,carrot,peas,garlic,soy sauce","{peas, carrot, garlic, onion, soy sauce, rice}"
1,Chickpea Curry,"chickpeas,onion,tomato,garlic,ginger,curry powder","{garlic, onion, curry powder, tomato, ginger, ..."
2,Paneer Butter Masala,"paneer,tomato,onion,butter,cream,garam masala","{onion, tomato, cream, garam masala, butter, p..."
3,Lemon Garlic Pasta,"pasta,garlic,lemon,olive oil,parsley","{garlic, olive oil, lemon, parsley, pasta}"
4,Vegetable Quinoa Bowl,"quinoa,cucumber,tomato,carrot,chickpeas,lemon","{carrot, cucumber, quinoa, tomato, lemon, chic..."


In [ ]:
# ----------------------------------------------
# 5. User input
# ----------------------------------------------
user_ingredients_text = "rice, carrot, peas, onion"
user_ingredients = ingredient_set(user_ingredients_text)
user_diet = "vegetarian"       # none, vegetarian, vegan, gluten-free, etc.
user_skill = "easy"             # easy, medium, hard
user_taste = "savory"
user_max_time = 40
strict_time_limit = False

user_question = f"""
I have {user_ingredients_text}. I am {user_diet},
my skill level is {user_skill}, I prefer {user_taste} food,
and I want a recipe within {user_max_time} minutes.
"""

In [ ]:
# ----------------------------------------------
# 6. Ingredient matching score
# ----------------------------------------------
user_set = ingredient_set(normalize_text(user_ingredients))

def ingredient_match(recipe_ingredients, user_ingredients_set):
    if not recipe_ingredients:
        return 0.0
    matched = recipe_ingredients.intersection(user_ingredients_set)
    return len(matched) / len(recipe_ingredients)

df["ingredient_score"] = df["ingredient_set"].apply(
    lambda x: ingredient_match(x, user_set)
)

In [ ]:
#Dietary filtering
def diet_tags(value):
    return {normalize_text(x).strip() for x in str(value).split(",") if normalize_text(x).strip()}

def dietary_compatible(tags_text, requested):
    requested = normalize_text(requested)
    tags = diet_tags(tags_text)

    if requested in ["", "none", "no restriction"]:
        return True
    if requested == "vegan":
        return "vegan" in tags
    if requested == "vegetarian":
        return "vegetarian" in tags or "vegan" in tags
    if requested in ["gluten-free", "gluten free"]:
        return "gluten-free" in tags or "gluten free" in tags
    return requested in tags

df["diet_compatible"] = df["dietary_tags"].apply(
    lambda x: dietary_compatible(x, user_diet)
)
filtered_df = df[df["diet_compatible"]].copy()
filtered_df["diet_score"] = 1.0

In [ ]:
# ----------------------------------------------
# 8. Skill-level compatibility
# ----------------------------------------------
skill_map = {
    "easy": 1, "beginner": 1,
    "medium": 2, "intermediate": 2,
    "hard": 3, "advanced": 3
}

def skill_score(recipe_level, user_level):
    r = skill_map.get(normalize_text(recipe_level), 2)
    u = skill_map.get(normalize_text(user_level), 2)
    return 1 - abs(r - u) / 2

def time_score(recipe_time, maximum):
    if pd.isna(recipe_time):
        return 0.0
    if recipe_time <= maximum:
        return 1.0
    return max(0.0, 1 - (recipe_time - maximum) / max(maximum, 1))

filtered_df["skill_score"] = filtered_df["difficulty"].apply(
    lambda x: skill_score(x, user_skill)
)
filtered_df["time_score"] = filtered_df["cooking_time_minutes"].apply(
    lambda x: time_score(x, user_max_time)
)

if strict_time_limit:
    filtered_df = filtered_df[
        filtered_df["cooking_time_minutes"] <= user_max_time
    ].copy()

In [ ]:
# ----------------------------------------------
# 9. TF-IDF taste and text similarity
# ----------------------------------------------
filtered_df["search_text"] = (
    filtered_df["ingredients"].apply(normalize_text) + " " +
    filtered_df["cuisine"].apply(normalize_text) + " " +
    filtered_df["taste"].apply(normalize_text) + " " +
    filtered_df["recipe_name"].apply(normalize_text)
)
vectorizer = TfidfVectorizer(stop_words="english")
recipe_matrix = vectorizer.fit_transform(filtered_df["search_text"])
user_vector = vectorizer.transform([
    user_ingredients_text + " " + user_taste + " " + user_question
])

filtered_df["taste_score"] = cosine_similarity(
    user_vector, recipe_matrix
).flatten()

In [ ]:
# ----------------------------------------------
# 10. Hybrid Ranking
# ----------------------------------------------
filtered_df["final_score"] = (
    0.40 * filtered_df["ingredient_score"] +
    0.20 * filtered_df["diet_score"] +
    0.15 * filtered_df["skill_score"] +
    0.15 * filtered_df["taste_score"] +
    0.10 * filtered_df["time_score"]
)

top_recipes = filtered_df.sort_values(
    "final_score", ascending=False
).head(10).copy()

display(top_recipes[[
    "recipe_id", "recipe_name", "difficulty",
    "cooking_time_minutes", "final_score"
]])

,recipe_id,recipe_name,difficulty,cooking_time_minutes,final_score
0,1,Vegetable Fried Rice,easy,25,0.810802
27,28,Egg Fried Rice,easy,20,0.796154
24,25,Vegetable Coconut Rice,medium,35,0.712699
15,16,Vegetable Upma,easy,25,0.694734
7,8,Masala Oats,easy,15,0.689988
11,12,Vegetable Poha,easy,20,0.688066
42,43,Vegetable Pulao,medium,35,0.639779
14,15,Mushroom Garlic Rice,easy,25,0.631471
40,41,Vegetable Dosa,medium,35,0.621395
47,48,Vegetable Sushi Bowl,easy,25,0.618944


In [ ]:
###Create RAG documents
def recipe_to_document(row):
    return f"""
Recipe ID: {row['recipe_id']}
Recipe Name: {row['recipe_name']}
Ingredients: {row['ingredients']}
Dietary Tags: {row['dietary_tags']}
Difficulty: {row['difficulty']}
Cooking Time: {row['cooking_time_minutes']} minutes
Cuisine: {row['cuisine']}
Taste: {row['taste']}
Calories per serving: {row['calories']}
Protein per serving: {row['protein_g']} g
Carbohydrates per serving: {row['carbs_g']} g
Fat per serving: {row['fat_g']} g
Instructions: {row['instructions']}
""".strip()

documents = [recipe_to_document(row) for _, row in top_recipes.iterrows()]

In [ ]:
## 13. Embeddings and FAISS
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Vectors stored:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vectors stored: 10


In [ ]:
#RAG Retrieval
query_embedding = embedding_model.encode(
    [user_question], normalize_embeddings=True
)
query_embedding = np.asarray(query_embedding, dtype="float32")

k = min(5, len(documents))
scores, indices = index.search(query_embedding, k)

retrieved_documents = [documents[i] for i in indices[0]]
context = "\\n\\n---\\n\\n".join(retrieved_documents)

for rank, score in enumerate(scores[0], start=1):
    print(f"Rank {rank}: {score:.4f}")

Rank 1: 0.6687
Rank 2: 0.6177
Rank 3: 0.6169
Rank 4: 0.6107
Rank 5: 0.6091


In [ ]:
#Grounded LLM Prompt
prompt = f"""
You are a grounded recipe recommendation assistant.
Use ONLY the retrieved recipe context below.

User request:
{user_question}

Retrieved context:
{context}

Tasks:
1. Recommend the most suitable recipes.
2. Explain the match using ingredients, diet, skill, taste, and time.
3. Show nutrition only when present in the context.
4. Give concise cooking tips based only on the instructions.
5. Do not invent ingredients, quantities, nutrition, cooking times,
   dietary classifications, recipe names, or health benefits.
6. Do not guarantee allergy safety or medical suitability.
7. If information is missing, say that the recipe knowledge base
   does not contain enough information to make that claim.
8. Treat nutrition as approximate unless professionally verified.
"""


In [37]:
from google import genai
from google.colab import userdata

# Get the Gemini API key from Colab secrets
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')

# Initialize the Google GenAI Client
client = genai.Client(api_key=GEMINI_API_KEY)

# Generate content using gemini-2.5-flash
response = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt,
    config={
        'system_instruction': 'You are a grounded recipe assistant. Do not invent facts.',
        'temperature': 0.2
    }
)

answer = response.text
print(answer)

Based on your request, here are the most suitable recipe recommendations from the provided context:

### Recommended Recipes

#### 1. Vegetable Fried Rice (Recipe ID: 1)
* **Match Explanation:** 
  * **Ingredients:** Uses your available ingredients (rice, onion, carrot, peas) plus garlic and soy sauce.
  * **Diet:** Vegetarian (and vegan).
  * **Skill:** Easy.
  * **Taste:** Savory and spicy.
  * **Time:** 25 minutes (well within your 40-minute limit).
* **Nutrition (Approximate):**
  * Calories per serving: 420
  * Protein per serving: 9 g
  * Carbohydrates per serving: 68 g
  * Fat per serving: 12 g
* **Cooking Tips:** Cook the rice first. Stir-fry the garlic and vegetables, then add the rice and soy sauce, tossing until hot.

#### 2. Egg Fried Rice (Recipe ID: 28)
* **Match Explanation:** 
  * **Ingredients:** Uses your available ingredients (rice, onion, carrot, peas) plus egg and soy sauce.
  * **Diet:** Vegetarian.
  * **Skill:** Easy.
  * **Taste:** Savory.
  * **Time:** 20 minu

In [38]:
#Nutrition and cooking tips
def display_nutrition(row):
    print("Recipe:", row["recipe_name"])
    print("Calories:", row["calories"])
    print("Protein:", row["protein_g"], "g")
    print("Carbs:", row["carbs_g"], "g")
    print("Fat:", row["fat_g"], "g")

if not top_recipes.empty:
    display_nutrition(top_recipes.iloc[0])

Recipe: Vegetable Fried Rice
Calories: 420
Protein: 9 g
Carbs: 68 g
Fat: 12 g


In [39]:
#Feedback Scehma
feedback_columns = [
    "user_id",
    "recipe_id",
    "rating",
    "comment",
    "timestamp"
]

feedback_df = pd.DataFrame(columns=feedback_columns)

In [40]:
from datetime import datetime

def add_feedback(
    feedback_df,
    user_id,
    recipe_id,
    rating,
    comment=""
):
    if not 1 <= int(rating) <= 5:
        raise ValueError("Rating must be between 1 and 5")

    new_row = pd.DataFrame([{
        "user_id": str(user_id),
        "recipe_id": str(recipe_id),
        "rating": int(rating),
        "comment": comment,
        "timestamp": datetime.now().isoformat()
    }])

    return pd.concat(
        [feedback_df, new_row],
        ignore_index=True
    )

In [41]:
feedback_df = add_feedback(
    feedback_df,
    user_id="user_001",
    recipe_id=str(top_recipes.iloc[0]["recipe_id"]),
    rating=5,
    comment="Easy and tasty"
)

feedback_df.to_csv("/content/sample_data/recipe_feedback.csv", index=False)


In [42]:
#Load existing feedback
import os

if os.path.exists("recipe_feedback.csv"):
    feedback_df = pd.read_csv("/content/sample_data/recipe_feedback.csv")
else:
    feedback_df = pd.DataFrame(columns=feedback_columns)


In [43]:
#Calculate the average rating for the selected user
# and each recipe
def get_user_recipe_ratings(feedback_df, user_id):
    if feedback_df.empty:
        return {}

    user_feedback = feedback_df[
        feedback_df["user_id"].astype(str) == str(user_id)
    ].copy()

    if user_feedback.empty:
        return {}

    user_feedback["rating"] = pd.to_numeric(
        user_feedback["rating"],
        errors="coerce"
    )

    user_feedback = user_feedback.dropna(subset=["rating"])

    user_recipe_ratings = user_feedback.groupby(
        "recipe_id"
    )["rating"].mean()

    return user_recipe_ratings.to_dict()

In [44]:
current_user_id = "user_001"

user_rating_map = get_user_recipe_ratings(
    feedback_df,
    current_user_id
)

filtered_df["user_rating"] = (
    filtered_df["recipe_id"].astype(str).map(user_rating_map)
)

In [47]:
#Convert Ratings into a User Feedback Score
#Ratings are normalized from 1–5 into a value between 0 and 1:
def normalize_rating(rating, default_rating=3.0):
    if pd.isna(rating):
        rating = default_rating

    return (float(rating) - 1.0) / 4.0

filtered_df["feedback_score"] = filtered_df[
    "user_rating"
].apply(normalize_rating)

In [48]:
#User-Specific Personalized Ranking
filtered_df["personalized_score"] = (
    0.35 * filtered_df["ingredient_score"]
    + 0.20 * filtered_df["diet_score"]
    + 0.15 * filtered_df["skill_score"]
    + 0.15 * filtered_df["taste_score"]
    + 0.10 * filtered_df["time_score"]
    + 0.05 * filtered_df["feedback_score"]
)

In [49]:
#Avoiding Repeated Ratings from the Same User
def get_latest_user_recipe_ratings(feedback_df, user_id):
    if feedback_df.empty:
        return {}

    user_feedback = feedback_df[
        feedback_df["user_id"].astype(str) == str(user_id)
    ].copy()

    if user_feedback.empty:
        return {}

    user_feedback["timestamp"] = pd.to_datetime(
        user_feedback["timestamp"],
        errors="coerce"
    )

    user_feedback["rating"] = pd.to_numeric(
        user_feedback["rating"],
        errors="coerce"
    )

    user_feedback = user_feedback.dropna(
        subset=["rating", "timestamp"]
    )

    latest_feedback = (
        user_feedback
        .sort_values("timestamp")
        .drop_duplicates(
            subset=["user_id", "recipe_id"],
            keep="last"
        )
    )

    return dict(
        zip(
            latest_feedback["recipe_id"].astype(str),
            latest_feedback["rating"]
        )
    )

In [50]:
#Learning Broader User Preferences
def get_liked_recipe_preferences(
    feedback_df,
    recipes_df,
    user_id,
    minimum_rating=4
):
    user_feedback = feedback_df[
        feedback_df["user_id"].astype(str) == str(user_id)
    ].copy()

    user_feedback["rating"] = pd.to_numeric(
        user_feedback["rating"],
        errors="coerce"
    )

    liked_ids = user_feedback.loc[
        user_feedback["rating"] >= minimum_rating,
        "recipe_id"
    ].astype(str)

    liked_recipes = recipes_df[
        recipes_df["recipe_id"].astype(str).isin(liked_ids)
    ]

    preferred_cuisines = (
        liked_recipes["cuisine"]
        .dropna()
        .astype(str)
        .str.lower()
        .value_counts()
        .to_dict()
    )

    preferred_tastes = (
        liked_recipes["taste"]
        .dropna()
        .astype(str)
        .str.lower()
        .value_counts()
        .to_dict()
    )

    return {
        "preferred_cuisines": preferred_cuisines,
        "preferred_tastes": preferred_tastes
    }

In [51]:
# Weekly meal planning
def create_weekly_meal_plan(ranked_df, days=7, meals_per_day=1):
    if ranked_df.empty:
        return pd.DataFrame()

    selected = ranked_df.drop_duplicates("recipe_id").head(days * meals_per_day).copy()
    selected["day"] = [
        f"Day {i // meals_per_day + 1}" for i in range(len(selected))
    ]
    return selected

weekly_plan = create_weekly_meal_plan(
    filtered_df.sort_values("final_score", ascending=False),
    days=7,
    meals_per_day=1
)
display(weekly_plan)

,recipe_id,recipe_name,ingredients,dietary_tags,difficulty,cooking_time_minutes,cuisine,taste,calories,protein_g,...,diet_score,skill_score,time_score,search_text,taste_score,final_score,user_rating,feedback_score,personalized_score,day
0,1,Vegetable Fried Rice,"rice,onion,carrot,peas,garlic,soy sauce","vegan,vegetarian",easy,25,Asian,"savory,spicy",420,9,...,1.0,1.0,1.0,"rice,onion,carrot,peas,garlic,soy sauce asian ...",0.627568,0.810802,NaN,0.5,0.802468,Day 1
27,28,Egg Fried Rice,"rice,egg,onion,carrot,peas,soy sauce",vegetarian,easy,20,Asian,savory,450,16,...,1.0,1.0,1.0,"rice,egg,onion,carrot,peas,soy sauce asian sav...",0.529915,0.796154,NaN,0.5,0.787821,Day 2
24,25,Vegetable Coconut Rice,"rice,coconut milk,carrot,peas,onion,cashew","vegan,vegetarian,gluten-free",medium,35,Indian,"mild,creamy",460,9,...,1.0,0.5,1.0,"rice,coconut milk,carrot,peas,onion,cashew ind...",0.473549,0.712699,NaN,0.5,0.704366,Day 3
15,16,Vegetable Upma,"semolina,onion,carrot,peas,cashew,mustard seeds","vegan,vegetarian",easy,25,Indian,"savory,mild",340,9,...,1.0,1.0,1.0,"semolina,onion,carrot,peas,cashew,mustard seed...",0.298226,0.694734,NaN,0.5,0.694734,Day 4
7,8,Masala Oats,"oats,onion,carrot,peas,tomato,green chili","vegan,vegetarian",easy,15,Indian,"spicy,savory",300,10,...,1.0,1.0,1.0,"oats,onion,carrot,peas,tomato,green chili indi...",0.266587,0.689988,NaN,0.5,0.689988,Day 5
11,12,Vegetable Poha,"poha,onion,peanuts,peas,carrot,mustard seeds","vegan,vegetarian,gluten-free",easy,20,Indian,"savory,mild",320,8,...,1.0,1.0,1.0,"poha,onion,peanuts,peas,carrot,mustard seeds i...",0.253776,0.688066,NaN,0.5,0.688066,Day 6
42,43,Vegetable Pulao,"basmati rice,carrot,peas,beans,onion,whole spices","vegan,vegetarian,gluten-free",medium,35,Indian,"aromatic,savory",430,9,...,1.0,0.5,1.0,"basmati rice,carrot,peas,beans,onion,whole spi...",0.431860,0.639779,NaN,0.5,0.639779,Day 7


In [52]:
# Shopping list generation
def shopping_list(recipe_ingredients, available_ingredients):
    required = ingredient_set(recipe_ingredients)
    available = ingredient_set(available_ingredients)
    return sorted(required - available)

print(shopping_list(
    "rice, carrot, peas, onion, garlic, soy sauce",
    "rice, carrot, peas"
))

['garlic', 'onion', 'soy sauce']


In [53]:
# Cooking-time optimization
def recommendation_weights(mode="balanced"):
    if mode == "fastest":
        return {
            "ingredient": 0.30, "diet": 0.25, "skill": 0.10,
            "taste": 0.10, "time": 0.25
        }
    return {
        "ingredient": 0.40, "diet": 0.20, "skill": 0.15,
        "taste": 0.15, "time": 0.10
    }

In [54]:
# Optional beverage pairings
# Use non-alcoholic options for a general-purpose application.
def beverage_pairing(cuisine, taste):
    cuisine = normalize_text(cuisine)
    taste = normalize_text(taste)

    if "indian" in cuisine or "spicy" in taste:
        return "Water, lemon water, or a yogurt-based drink"
    if "italian" in cuisine:
        return "Sparkling water or a citrus mocktail"
    if "asian" in cuisine:
        return "Green tea or chilled lemon water"
    return "Water or a non-alcoholic fruit drink"